<a href="https://colab.research.google.com/github/MarceAdones/Dancegdp-dashboard/blob/main/DanceManager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import csv

# Configuración inicial: Cursos disponibles y carpeta de almacenamiento
CURSOS = ["Ballet", "Salsa", "Bachata", "Danza Urbana", "K-Pop Dance", "Contemporaneo"]
CARPETA_CURSOS = "cursos"

def inicializar_sistema():
    """Crea la carpeta de cursos y los archivos CSV si no existen."""
    if not os.path.exists(CARPETA_CURSOS):
        os.makedirs(CARPETA_CURSOS)

    for curso in CURSOS:
        nombre_archivo = os.path.join(CARPETA_CURSOS, f"{curso.lower().replace(' ', '_')}.csv")
        if not os.path.exists(nombre_archivo):
            with open(nombre_archivo, mode='w', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                # Encabezados del archivo CSV
                writer.writerow(["ID", "Nombre", "Apellido", "Edad"])

def obtener_ruta_curso(curso):
    """Devuelve la ruta del archivo CSV de un curso específico."""
    return os.path.join(CARPETA_CURSOS, f"{curso.lower().replace(' ', '_')}.csv")

def leer_estudiantes(curso):
    """Lee y retorna la lista de estudiantes de un curso."""
    ruta = obtener_ruta_curso(curso)
    estudiantes = []
    if os.path.exists(ruta):
        with open(ruta, mode='r', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            for row in reader:
                estudiantes.append(row)
    return estudiantes

def guardar_estudiantes(curso, estudiantes):
    """Guarda la lista completa de estudiantes en el archivo CSV del curso."""
    ruta = obtener_ruta_curso(curso)
    with open(ruta, mode='w', newline='', encoding='utf-8') as file:
        fieldnames = ["ID", "Nombre", "Apellido", "Edad"]
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for est in estudiantes:
            writer.writerow(est)

# ==================== FUNCIONALIDADES PRINCIPALES ====================

def mostrar_cursos():
    print("\n🩰 CURSOS DISPONIBLES EN DANCEMANAGER:")
    for i, curso in enumerate(CURSOS, 1):
        cant = len(leer_estudiantes(curso))
        print(f"{i}. {curso} ({cant} estudiantes)")

def administrar_estudiantes():
    mostrar_cursos()
    try:
        opc = int(input("\nSelecciona el número del curso a administrar: ")) - 1
        if opc < 0 or opc >= len(CURSOS):
            print("❌ Opción inválida.")
            return
        curso_sel = CURSOS[opc]
    except ValueError:
        print("❌ Por favor, ingresa un número válido.")
        return

    while True:
        print(f"\n--- 👥 ADMINISTRACIÓN DE: {curso_sel.upper()} ---")
        print("1. Visualizar lista de alumnos")
        print("2. Agregar nuevo estudiante")
        print("3. Eliminar estudiante")
        print("4. Volver al menú principal")

        accion = input("Selecciona una opción: ")
        estudiantes = leer_estudiantes(curso_sel)

        if accion == "1":
            print(f"\n📋 LISTA DE ESTUDIANTES - {curso_sel}:")
            if not estudiantes:
                print("No hay estudiantes inscritos en este curso.")
            else:
                print(f"{'ID':<6} | {'Nombre':<15} | {'Apellido':<15} | {'Edad':<5}")
                print("-" * 50)
                for est in estudiantes:
                    print(f"{est['ID']:<6} | {est['Nombre']:<15} | {est['Apellido']:<15} | {est['Edad']:<5}")

        elif accion == "2":
            print("\n📝 AGREGAR ESTUDIANTE:")
            id_est = input("Introduce un ID único o Rut: ")
            if any(est['ID'] == id_est for est in estudiantes):
                print("❌ Error: Ya existe un estudiante con ese ID en este curso.")
                continue
            nombre = input("Nombre: ").strip()
            apellido = input("Apellido: ").strip()
            edad = input("Edad: ").strip()

            estudiantes.append({"ID": id_est, "Nombre": nombre, "Apellido": apellido, "Edad": edad})
            guardar_estudiantes(curso_sel, estudiantes)
            print(f"✅ {nombre} {apellido} ha sido agregado exitosamente.")

        elif accion == "3":
            print("\n❌ ELIMINAR ESTUDIANTE:")
            id_eliminar = input("Introduce el ID del estudiante a eliminar: ")
            nuevo_listado = [est for est in estudiantes if est['ID'] != id_eliminar]

            if len(nuevo_listado) == len(estudiantes):
                print("❌ No se encontró ningún estudiante con ese ID.")
            else:
                guardar_estudiantes(curso_sel, nuevo_listado)
                print("✅ Estudiante eliminado correctamente.")

        elif accion == "4":
            break
        else:
            print("❌ Opción no válida.")

def buscar_estudiantes():
    print("\n🔍 BÚSQUEDA DE ESTUDIANTES:")
    print("1. Buscar en un curso específico")
    print("2. Buscar en TODOS los cursos")
    opc_busqueda = input("Selecciona una opción: ")

    termino = input("Introduce el Nombre o Apellido a buscar: ").lower().strip()
    resultados = []

    cursos_a_buscar = CURSOS
    if opc_busqueda == "1":
        mostrar_cursos()
        try:
            idx = int(input("Selecciona el número del curso: ")) - 1
            if 0 <= idx < len(CURSOS):
                cursos_a_buscar = [CURSOS[idx]]
            else:
                print("❌ Curso no válido.")
                return
        except ValueError:
            print("❌ Entrada inválida.")
            return

    for curso in cursos_a_buscar:
        lista = leer_estudiantes(curso)
        for est in lista:
            if termino in est['Nombre'].lower() or termino in est['Apellido'].lower():
                resultados.append((curso, est))

    print(f"\n🎯 RESULTADOS DE BÚSQUEDA ('{termino}'):")
    if not resultados:
        print("No se encontraron estudiantes.")
    else:
        print(f"{'Curso':<15} | {'ID':<6} | {'Nombre':<15} | {'Apellido':<15} | {'Edad':<5}")
        print("-" * 65)
        for curso, est in resultados:
            print(f"{curso:<15} | {est['ID']:<6} | {est['Nombre']:<15} | {est['Apellido']:<15} | {est['Edad']:<5}")

def generar_estadisticas():
    print("\n📊 ESTADÍSTICAS Y REPORTES:")
    total_general = 0
    curso_mas = ""
    curso_menos = ""
    max_est = -1
    min_est = 999999

    print("\nCantidad de estudiantes por curso:")
    print("-" * 35)
    for curso in CURSOS:
        cant = len(leer_estudiantes(curso))
        total_general += cant
        print(f" * {curso:<15}: {cant} alumnos")

        if cant > max_est:
            max_est = cant
            curso_mas = curso
        if cant < min_est:
            min_est = cant
            curso_menos = curso

    print("-" * 35)
    print(f"👥 Cantidad TOTAL de estudiantes: {total_general}")
    if total_general > 0:
        print(f"🔥 Curso con MÁS estudiantes: {curso_mas} ({max_est} alumnos)")
        print(f"❄️ Curso con MENOS estudiantes: {curso_menos} ({min_est} alumnos)")

# ==================== MENÚ PRINCIPAL ====================

def menu_principal():
    inicializar_sistema()
    while True:
        print("\n======================================")
        print("      💃 BIENVENIDO A DANCEMANAGER    ")
        print("======================================")
        print("1. Ver Cursos Disponibles")
        print("2. Administrar Estudiantes (Ver/Añadir/Eliminar)")
        print("3. Buscar Estudiante")
        print("4. Ver Estadísticas y Reportes")
        print("5. Salir del Sistema")
        print("======================================")

        opcion = input("Por favor, elija una opción (1-5): ")

        if opcion == "1":
            mostrar_cursos()
        elif opcion == "2":
            administrar_estudiantes()
        elif opcion == "3":
            buscar_estudiantes()
        elif opcion == "4":
            generar_estadisticas()
        elif opcion == "5":
            print("\n¡Gracias por usar DanceManager! ¡Sigue bailando! 💃✨\n")
            break
        else:
            print("❌ Opción inválida. Intenta nuevamente.")

if __name__ == "__main__":
    menu_principal()


      💃 BIENVENIDO A DANCEMANAGER    
1. Ver Cursos Disponibles
2. Administrar Estudiantes (Ver/Añadir/Eliminar)
3. Buscar Estudiante
4. Ver Estadísticas y Reportes
5. Salir del Sistema

📊 ESTADÍSTICAS Y REPORTES:

Cantidad de estudiantes por curso:
-----------------------------------
 * Ballet         : 0 alumnos
 * Salsa          : 0 alumnos
 * Bachata        : 0 alumnos
 * Danza Urbana   : 0 alumnos
 * K-Pop Dance    : 0 alumnos
 * Contemporaneo  : 0 alumnos
-----------------------------------
👥 Cantidad TOTAL de estudiantes: 0

      💃 BIENVENIDO A DANCEMANAGER    
1. Ver Cursos Disponibles
2. Administrar Estudiantes (Ver/Añadir/Eliminar)
3. Buscar Estudiante
4. Ver Estadísticas y Reportes
5. Salir del Sistema
